## Section 3 — Exploration des Donnees

L'exploration est une etape fondamentale. Elle permet de :
- Comprendre la structure et les caracteristiques du dataset
- Detecter les problemes (desequilibre, images corrompues)
- Visualiser les differences entre les classes
- Justifier les choix de pretraitement

### Ce qu'on analyse
1. Distribution des classes — desequilibre quantitatif
2. Caracteristiques visuelles — differences NORMAL vs PNEUMONIA
3. Distribution des pixels — intensites et contrastes
4. Effet du CLAHE — avant/apres pretraitement

In [ ]:
# ============================================================
# SECTION 3 : EXPLORATION DES DONNEES
# ============================================================

fixer_seed(42)

# ────────────────────────────────────────────────────────────
# 3.1 Distribution des classes
# ────────────────────────────────────────────────────────────
# Objectif : mesurer le desequilibre entre NORMAL et PNEUMONIA
# Un desequilibre important biaise l'apprentissage du modele

print("3.1 DISTRIBUTION DES CLASSES")
print("="*45)

stats = {}
for split in ["train", "val", "test"]:
    stats[split] = {}
    for classe in ["NORMAL", "PNEUMONIA"]:
        path = os.path.join(DATA_DIR, split, classe)
        stats[split][classe] = len(os.listdir(path))

for split, valeurs in stats.items():
    total = sum(valeurs.values())
    print(f"\n{split.upper()} ({total} images) :")
    for classe, nb in valeurs.items():
        pct = nb / total * 100
        barre = "█" * int(pct / 3)
        print(f"  {classe:<12} : {nb:>5} ({pct:.1f}%) {barre}")

ratio = stats["train"]["PNEUMONIA"] / stats["train"]["NORMAL"]
print(f"\nRatio desequilibre : 1:{ratio:.2f}")
print(f"Un modele naif (toujours PNEUMONIA) = {stats['train']['PNEUMONIA']/(stats['train']['NORMAL']+stats['train']['PNEUMONIA'])*100:.1f}% accuracy")
print("=> Ce modele naif serait inutile cliniquement !")

# ── Graphique de distribution ─────────────────────────────────
fig, axes = plt.subplots(1, 3, figsize=(15, 5))
fig.suptitle("Distribution des Classes par Split", fontsize=14, fontweight="bold")
couleurs = {"NORMAL": "steelblue", "PNEUMONIA": "tomato"}

for idx, split in enumerate(["train", "val", "test"]):
    valeurs = stats[split]
    total   = sum(valeurs.values())
    bars    = axes[idx].bar(
        valeurs.keys(), valeurs.values(),
        color=[couleurs[k] for k in valeurs.keys()],
        width=0.5, edgecolor="white", linewidth=1.5
    )
    axes[idx].set_title(f"{split.upper()} ({total} images)", fontweight="bold")
    axes[idx].set_ylabel("Nombre d'images")
    for bar, (classe, nb) in zip(bars, valeurs.items()):
        pct = nb / total * 100
        axes[idx].text(bar.get_x()+bar.get_width()/2,
                       bar.get_height()+30,
                       f"{nb}\n({pct:.0f}%)",
                       ha="center", fontsize=10, fontweight="bold")
    axes[idx].set_ylim(0, max(valeurs.values())*1.3)

plt.tight_layout()
plt.show()

# ────────────────────────────────────────────────────────────
# 3.2 Visualisation des radiographies
# ────────────────────────────────────────────────────────────
# Objectif : observer visuellement les differences entre les classes
# NORMAL     : poumons clairs, pas d'opacite, structure symetrique
# PNEUMONIA  : zones blanches (infiltrats), opacites, asymetrie possible

print("\n3.2 VISUALISATION DES RADIOGRAPHIES")

fig, axes = plt.subplots(2, 6, figsize=(20, 7))
fig.suptitle("Exemples de Radiographies Pulmonaires", fontsize=13, fontweight="bold")

for i, classe in enumerate(["NORMAL", "PNEUMONIA"]):
    dossier = os.path.join(DATA_DIR, "train", classe)
    images  = random.sample(os.listdir(dossier), 6)
    couleur = "steelblue" if classe == "NORMAL" else "tomato"
    for j, nom in enumerate(images):
        img = Image.open(os.path.join(dossier, nom)).convert("L")
        axes[i][j].imshow(img, cmap="gray")
        axes[i][j].axis("off")
        if j == 0:
            axes[i][j].set_title(classe, color=couleur,
                                  fontweight="bold", fontsize=11, loc="left")

# Annotations medicales
fig.text(0.01, 0.75, "Poumons clairs\nSymetriques\nPas d'opacite",
         fontsize=9, color="steelblue", va="center",
         bbox=dict(boxstyle="round", facecolor="lightblue", alpha=0.3))
fig.text(0.01, 0.28, "Opacites\nInfiltrats\nZones blanches",
         fontsize=9, color="tomato", va="center",
         bbox=dict(boxstyle="round", facecolor="lightyellow", alpha=0.3))
plt.tight_layout()
plt.show()

print("Observation : les radiographies PNEUMONIA presentent des zones blanches")
print("(infiltrats) dans les poumons. C'est ce que le CNN doit apprendre a detecter.")

# ────────────────────────────────────────────────────────────
# 3.3 Analyse des intensites de pixels
# ────────────────────────────────────────────────────────────
# Objectif : confirmer que les deux classes ont des distributions
# de pixels differentes, ce qui justifie l'utilisation du Deep Learning

print("\n3.3 ANALYSE DES INTENSITES DE PIXELS")

fig, axes = plt.subplots(1, 2, figsize=(14, 5))
fig.suptitle("Distribution des Intensites de Pixels", fontsize=13, fontweight="bold")

stats_pixels = {}
for classe, couleur in [("NORMAL","steelblue"), ("PNEUMONIA","tomato")]:
    dossier  = os.path.join(DATA_DIR, "train", classe)
    fichiers = random.sample(os.listdir(dossier), 50)
    tous_pixels, moyennes = [], []
    for f in fichiers:
        img = cv2.imread(os.path.join(dossier, f), cv2.IMREAD_GRAYSCALE)
        img = cv2.resize(img, (224, 224))
        tous_pixels.extend(img.flatten().tolist())
        moyennes.append(img.mean())
    tous_pixels = np.array(tous_pixels)
    stats_pixels[classe] = {
        "moyenne": np.mean(tous_pixels),
        "ecart_type": np.std(tous_pixels),
        "mediane": np.median(tous_pixels)
    }
    axes[0].hist(tous_pixels, bins=50, alpha=0.5, color=couleur,
                 label=classe, density=True)
    axes[1].hist(moyennes, bins=20, alpha=0.6, color=couleur,
                 label=classe, edgecolor="white")

axes[0].set_title("Distribution globale des pixels")
axes[0].set_xlabel("Intensite (0=noir, 255=blanc)")
axes[0].set_ylabel("Densite")
axes[0].legend(); axes[0].grid(True, alpha=0.3)
axes[0].axvline(x=127, color="black", linestyle="--", alpha=0.5, label="Milieu")

axes[1].set_title("Intensite moyenne par image")
axes[1].set_xlabel("Intensite moyenne")
axes[1].set_ylabel("Nombre d'images")
axes[1].legend(); axes[1].grid(True, alpha=0.3)

plt.tight_layout(); plt.show()

print("\nStatistiques des pixels :")
for classe, s in stats_pixels.items():
    print(f"  {classe} : moyenne={s['moyenne']:.1f} | ecart-type={s['ecart_type']:.1f} | mediane={s['mediane']:.1f}")
print("\nInterpretation : les radios PNEUMONIA ont plus de pixels a haute intensite")
print("(zones blanches = infiltrats) que les radios NORMAL (zones sombres = air).")

# ────────────────────────────────────────────────────────────
# 3.4 Effet du CLAHE
# ────────────────────────────────────────────────────────────
# CLAHE = Contrast Limited Adaptive Histogram Equalization
# Ameliore le contraste LOCAL de l'image zone par zone
# Fait ressortir les infiltrats sans surexposer les zones claires

print("\n3.4 EFFET DU CLAHE")
print("""
CLAHE ameliore le contraste local de chaque zone de l'image.
Contrairement a l'egalisation globale, il evite de surexposer
les zones deja claires tout en faisant ressortir les infiltrats.""")

fig, axes = plt.subplots(2, 4, figsize=(18, 9))
fig.suptitle("Effet du CLAHE — Avant / Apres", fontsize=13, fontweight="bold")

for i, classe in enumerate(["NORMAL", "PNEUMONIA"]):
    dossier = os.path.join(DATA_DIR, "train", classe)
    images  = random.sample(os.listdir(dossier), 4)
    couleur = "steelblue" if classe == "NORMAL" else "tomato"
    for j, nom in enumerate(images):
        chemin    = os.path.join(dossier, nom)
        img_brute = cv2.imread(chemin, cv2.IMREAD_GRAYSCALE)
        img_brute = cv2.resize(img_brute, (224, 224))
        clahe     = cv2.createCLAHE(clipLimit=2.0, tileGridSize=(8, 8))
        img_clahe = clahe.apply(img_brute)
        if j < 2:
            axes[i][j*2].imshow(img_brute, cmap="gray")
            axes[i][j*2].set_title(f"{classe}\nOriginal", color=couleur, fontsize=9)
            axes[i][j*2].axis("off")
            axes[i][j*2+1].imshow(img_clahe, cmap="gray")
            axes[i][j*2+1].set_title(f"{classe}\nApres CLAHE", color=couleur, fontsize=9)
            axes[i][j*2+1].axis("off")

plt.tight_layout(); plt.show()
print("Observation : apres CLAHE, les infiltrats sont plus visibles sur les radios PNEUMONIA.")